In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [2]:
df = pd.read_csv('label_data.csv', sep=';' )
df.head()

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,Artist (Ind.),# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm,Popular
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5.176,0.033,0.483,0.000,...,Eslabon Armado,Nationality 1,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,0.849862,1
1,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4.019,0.061,0.143,0.629,...,Bad Bunny,Nationality 1,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,0.883423,1
2,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5.678,0.333,0.213,0.000,...,Yng Lvcas,Nationality 1,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,0.835301,1
3,4,Cupid - Twin Ver.,FIFTY FIFTY,2023-05-29,0.783,0.592,-8.332,0.033,0.435,0.000,...,FIFTY FIFTY,Nationality 1,South Korea,Asia,197,197.0,7FbrGaHYVDmfr7KoLIZnQ7,https://open.spotify.com/track/7FbrGaHYVDmfr7K...,0.758318,1
4,5,un x100to,"Grupo Frontera, Bad Bunny",2023-05-29,0.569,0.724,-4.076,0.047,0.228,0.000,...,Grupo Frontera,Nationality 1,Mexico,Latin-America,196,98.0,6pD0ufEQq0xdHSsRbg9LBK,https://open.spotify.com/track/6pD0ufEQq0xdHSs...,0.881769,1


In [3]:
df.shape

(9161, 22)

In [9]:
# X = df[['Danceability','Loudness_norm','Speechiness','Acousticness','Energy','Instrumentalness','Valence']]
X = df[['Loudness_norm','Acousticness','Energy','Instrumentalness']]
y = df['Popular']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42, stratify=y)

In [4]:
print("number of X_train :",X_train.shape[0])
print("number of X_test :",X_test.shape[0])
print("number of y_train :",y_train.shape[0])
print("number of y_test :",y_test.shape[0])

number of X_train : 6870
number of X_test : 2291
number of y_train : 6870
number of y_test : 2291


In [15]:
model = LogisticRegression(max_iter=1000, class_weight='balanced',random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [16]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5438673068529026

Confusion Matrix:
 [[908 783]
 [262 338]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.54      0.63      1691
           1       0.30      0.56      0.39       600

    accuracy                           0.54      2291
   macro avg       0.54      0.55      0.51      2291
weighted avg       0.65      0.54      0.57      2291



In [12]:
print(roc_auc_score(y_test, y_pred))

0.5526676829268293


In [14]:
coef = pd.Series(model.coef_[0], index=X.columns)
coef.sort_values(ascending=False)

Loudness_norm       1.623040
Danceability        0.623727
Acousticness        0.126595
Valence            -0.063481
Speechiness        -0.461768
Instrumentalness   -0.735826
Energy             -1.079389
dtype: float64

In [8]:
pipe = Pipeline([
    ('clf', LogisticRegression(max_iter=2000, random_state=42))
])


param_grid = {
    'clf__C': [0.01, 0.1, 0.5, 1, 2, 5, 10],
    'clf__class_weight': [None, 'balanced'],
    'clf__solver': ['saga', 'liblinear'],
    'clf__penalty': ['l1']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=cv,
    n_jobs=-1,
    scoring='f1',
    refit=True,
    verbose=2
)
grid.fit(X_train, y_train)

print("\n Best Parameters:")
print(grid.best_params_)
print(f"Best CV F1 Score: {grid.best_score_:.3f}")

Fitting 5 folds for each of 28 candidates, totalling 140 fits
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=liblinear; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=saga; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=liblinear; total time=   0.0s
[CV] END clf__C=0.01, clf__class_weight=None, clf__penalty=l1, clf__solver=liblinear; total time=   0.0s
[CV] END cl

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\n Test Set Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


 Test Set Results:
Accuracy: 0.5450081833060556
F1-score: 0.39565217391304347

Confusion Matrix:
 [[726 627]
 [207 273]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.54      0.64      1353
           1       0.30      0.57      0.40       480

    accuracy                           0.55      1833
   macro avg       0.54      0.55      0.52      1833
weighted avg       0.65      0.55      0.57      1833



In [6]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===========================================
# 3️⃣ Define Models
# ===========================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric='logloss',      
    )
}

# ===========================================
# 4️⃣ Parameter Grids for Tuning
# ===========================================
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 0.5, 1, 2, 5],
        "solver": ["lbfgs", "liblinear"]
    },
    "Random Forest": {
        "n_estimators": [200, 400],
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 5, 10]
    },
    "SVM": {
        "C": [0.5, 1, 2, 5, 10],
        "gamma": ["scale", 0.1, 0.01]
    },
    "XGBoost": {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "scale_pos_weight": [(len(y_train)-sum(y_train))/sum(y_train)]  
    }
}

# ===========================================
# 5️⃣ Run GridSearch for Each Model
# ===========================================
summary = []
best_models = {}

for name, model in models.items():
    print(f"\n===== 🔍 Tuning {name} =====")
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring='f1',          
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Best Params for {name}: {grid.best_params_}")
    print(f"CV Best F1: {grid.best_score_:.3f}")
    print(f"Test Accuracy: {acc:.3f}, Test F1: {f1:.3f}")

    summary.append({
        "Model": name,
        "CV_best_F1": grid.best_score_,
        "Test_Accuracy": acc,
        "Test_F1": f1
    })

# ===========================================
# 6️⃣ Summarize All Results
# ===========================================
summary_df = pd.DataFrame(summary).sort_values(by="Test_F1", ascending=False)
print("\n📊 Model Comparison:")
print(summary_df)


===== 🔍 Tuning Logistic Regression =====
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params for Logistic Regression: {'C': 0.5, 'solver': 'liblinear'}
CV Best F1: 0.373
Test Accuracy: 0.542, Test F1: 0.388

===== 🔍 Tuning Random Forest =====
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params for Random Forest: {'max_depth': 5, 'min_samples_split': 10, 'n_estimators': 200}
CV Best F1: 0.393
Test Accuracy: 0.529, Test F1: 0.427

===== 🔍 Tuning SVM =====
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Params for SVM: {'C': 10, 'gamma': 'scale'}
CV Best F1: 0.399
Test Accuracy: 0.503, Test F1: 0.414

===== 🔍 Tuning XGBoost =====
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Params for XGBoost: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'scale_pos_weight': 2.820646506777894, 'subsample': 1.0}
CV Best F1: 0.389
Test Accuracy: 0.547, Test F1: 0.425

📊 Model Compariso

In [8]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===========================================
# 3️⃣ Define Models
# ===========================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric='logloss',      
    ),
    "KNN": KNeighborsClassifier(n_neighbors = 5,  metric='minkowski', p =2),
    "GaussianNB": GaussianNB()
    
}

# ===========================================
# 4️⃣ Parameter Grids for Tuning
# ===========================================
param_grids = {
     "Logistic Regression": [
        # --- L2 group ---
        {
            "penalty": ["l2"],
            "solver": ["lbfgs", "liblinear","saga"],
            "C": [0.01, 0.1, 0.5, 1, 2, 5],
            "class_weight": ["balanced"]
        },
        # --- L1 group ---
        {
            "penalty": ["l1"],
            "solver": ["liblinear", "saga"],
            "C": [0.01, 0.1, 0.5, 1, 2, 5],
            "class_weight": ["balanced"]
        }
    ],
    "Random Forest": {
        "n_estimators": [200, 400],
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 5, 10]
    },
    "SVM": {
        "C": [0.5, 1, 2, 5, 10],
        "gamma": ["scale", 0.1, 0.01]
    },
    "XGBoost": {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "scale_pos_weight": [(len(y_train)-sum(y_train))/sum(y_train)]  
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9, 11, 15],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "GaussianNB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    }
}

# ===========================================
# 5️⃣ Run GridSearch for Each Model
# ===========================================
summary = []
best_models = {}

for name, model in models.items():
    print(f"\n===== 🔍 Tuning {name} =====")
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring='f1',          
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Best Params for {name}: {grid.best_params_}")
    print(f"CV Best F1: {grid.best_score_:.3f}")
    print(f"Test Accuracy: {acc:.3f}, Test F1: {f1:.3f}")
    
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print("\nConfusion Matrix [Actual rows, Predicted cols]:")
    print(pd.DataFrame(cm,
                       index=['Actual 0','Actual 1'],
                       columns=['Pred 0','Pred 1']))
    print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

    print("\nClassification Report:\n",
          classification_report(y_test, y_pred, digits=3))
    
    if hasattr(best_model, 'predict_proba'):
        y_pred_proba = best_model.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        print(f"ROC AUC Score: {roc_auc:.4f}")
    else:
        print("ROC AUC Score: Not available (model does not provide probabilities directly).")
    print(f"{'='*50}\n")




   
    summary.append({
        "Model": name,
        "CV_best_F1": grid.best_score_,
        "Test_Accuracy": acc,
        "Test_F1": f1,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })

# ===========================================
# 6️⃣ Summarize All Results
# ===========================================
summary_df = pd.DataFrame(summary).sort_values(by="Test_F1", ascending=False)
print("\n📊 Model Comparison:")
print(summary_df)


===== 🔍 Tuning Logistic Regression =====
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Params for Logistic Regression: {'C': 0.1, 'class_weight': 'balanced', 'penalty': 'l1', 'solver': 'liblinear'}
CV Best F1: 0.304
Test Accuracy: 0.515, Test F1: 0.308

Confusion Matrix [Actual rows, Predicted cols]:
          Pred 0  Pred 1
Actual 0     934     966
Actual 1     144     247
TN=934  FP=966  FN=144  TP=247

Classification Report:
               precision    recall  f1-score   support

           0      0.866     0.492     0.627      1900
           1      0.204     0.632     0.308       391

    accuracy                          0.515      2291
   macro avg      0.535     0.562     0.468      2291
weighted avg      0.753     0.515     0.573      2291

ROC AUC Score: 0.5864


===== 🔍 Tuning Random Forest =====
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params for Random Forest: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 400}
CV B

/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [10]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ===========================================
# 3️⃣ Define Models
# ===========================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric='logloss',      
    ),
    "KNN": KNeighborsClassifier(n_neighbors = 5,  metric='minkowski', p =2),
    "GaussianNB": GaussianNB()
    
}

# ===========================================
# 4️⃣ Parameter Grids for Tuning
# ===========================================
param_grids = {
     "Logistic Regression": [
        # --- L2 group ---
        {
            "penalty": ["l2"],
            "solver": ["lbfgs", "liblinear","saga"],
            "C": [0.01, 0.1, 0.5, 1, 2, 5],
            "class_weight": ["balanced"]
        },
        # --- L1 group ---
        {
            "penalty": ["l1"],
            "solver": ["liblinear", "saga"],
            "C": [0.01, 0.1, 0.5, 1, 2, 5],
            "class_weight": ["balanced"]
        }
    ],
    "Random Forest": {
        "n_estimators": [200, 400],
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 5, 10]
    },
    "SVM": {
        "C": [0.5, 1, 2, 5, 10],
        "gamma": ["scale", 0.1, 0.01]
    },
    "XGBoost": {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5, 7],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "scale_pos_weight": [(len(y_train)-sum(y_train))/sum(y_train)]  
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9, 11, 15],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "GaussianNB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    }
}

# ===========================================
# 5️⃣ Run GridSearch for Each Model
# ===========================================
summary = []
best_models = {}

for name, model in models.items():
    print(f"\n===== 🔍 Tuning {name} =====")
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=cv,
        scoring='f1',          
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_models[name] = best_model

    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Best Params for {name}: {grid.best_params_}")
    print(f"CV Best F1: {grid.best_score_:.3f}")
    print(f"Test Accuracy: {acc:.3f}, Test F1: {f1:.3f}")
    
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print("\nConfusion Matrix [Actual rows, Predicted cols]:")
    print(pd.DataFrame(cm,
                       index=['Actual 0','Actual 1'],
                       columns=['Pred 0','Pred 1']))
    print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

    print("\nClassification Report:\n",
          classification_report(y_test, y_pred, digits=3))
    
    if hasattr(best_model, 'predict_proba'):
        y_pred_proba = best_model.predict_proba(X_test)[:, 1] # Probability of the positive class (1)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        print(f"ROC AUC Score: {roc_auc:.4f}")
    else:
        print("ROC AUC Score: Not available (model does not provide probabilities directly).")
    print(f"{'='*50}\n")




   
    summary.append({
        "Model": name,
        "CV_best_F1": grid.best_score_,
        "Test_Accuracy": acc,
        "Test_F1": f1,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })

# ===========================================
# 6️⃣ Summarize All Results
# ===========================================
summary_df = pd.DataFrame(summary).sort_values(by="Test_F1", ascending=False)
print("\n📊 Model Comparison:")
print(summary_df)


===== 🔍 Tuning Logistic Regression =====
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Params for Logistic Regression: {'C': 1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
CV Best F1: 0.286
Test Accuracy: 0.532, Test F1: 0.279

Confusion Matrix [Actual rows, Predicted cols]:
          Pred 0  Pred 1
Actual 0    1010     890
Actual 1     183     208
TN=1010  FP=890  FN=183  TP=208

Classification Report:
               precision    recall  f1-score   support

           0      0.847     0.532     0.653      1900
           1      0.189     0.532     0.279       391

    accuracy                          0.532      2291
   macro avg      0.518     0.532     0.466      2291
weighted avg      0.734     0.532     0.589      2291

ROC AUC Score: 0.5632


===== 🔍 Tuning Random Forest =====
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params for Random Forest: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}
CV Best F

/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/AML-proj/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is